# Parte 2: Del LLM al clasificador
**Orden de trabajo:** el split se hace primero (aunque en el enunciado es el paso 3) para que el zero-shot, el LLM y el Transformer se evalúen sobre el mismo conjunto de test, que no se toca hasta la evaluación final.

## 1. Partición de datos 70/15/15 (paso 3 del enunciado)
Split estratificado por `etiqueta` con `random_state=42`.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv("datos/noticias.csv")
LABELS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

train, temp = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["etiqueta"], random_state=42)
print(len(train), len(val), len(test))

280 60 60


### 1.1 Noticias por categoría en cada split

In [2]:
pd.DataFrame({"train": train["etiqueta"].value_counts(),
              "val": val["etiqueta"].value_counts(),
              "test": test["etiqueta"].value_counts()})

,train,val,test
etiqueta,,,
ciencia_tecnologia,70,15,15
deportes,70,15,15
economia,70,15,15
politica,70,15,15


## 2. Línea base zero-shot (paso 1 del enunciado)
40 titulares del test clasificados con `Recognai/bert-base-spanish-wwm-cased-xnli`, sin fine-tuning. Este F1 macro es el piso mínimo a superar.

In [3]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli")
candidatas = {"política": "politica", "economía": "economia",
              "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}

muestra = test.sample(40, random_state=42)
salidas = zs(muestra["titular"].tolist(), candidate_labels=list(candidatas),
             hypothesis_template="Esta noticia trata sobre {}.")
muestra["pred_zs"] = [candidatas[o["labels"][0]] for o in salidas]

acc_zs = accuracy_score(muestra["etiqueta"], muestra["pred_zs"])
f1_zs = f1_score(muestra["etiqueta"], muestra["pred_zs"], average="macro")
print(f"Zero-shot -> Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f}")

/opt/anaconda3/envs/tarea1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 23:26:23.981000 10892 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13694.63it/s]


Zero-shot -> Accuracy: 0.575 | F1 macro: 0.571


## 3. Supervisión débil con LLM (paso 2 del enunciado)
Gemini etiqueta el dataset completo con un prompt estructurado. El resultado se guarda en `datos/etiquetas_llm.csv`.

In [6]:
import os, json, time, getpass
from datetime import date
from google import genai

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Pega tu clave de Gemini: ")   # no se guarda en el notebook
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO_LLM = "gemini-3.8-flash"   # gemini-2.5-flash ya no está disponible para claves nuevas (error 404)
# Para ver qué modelos tienes: [m.name for m in client.models.list()]

PROMPT = """Eres un editor de un diario peruano. Clasifica cada titular en UNA categoría:
politica, economia, deportes, ciencia_tecnologia.
Devuelve SOLO un JSON: lista de objetos {{"id": int, "etiqueta": str, "confianza": float entre 0 y 1}}.

Titulares:
{titulares}"""

from google.genai import errors

def etiquetar_lote(lote, intentos=5):
    """Llama a Gemini; si está saturado (503) o se pasa del límite por minuto (429), espera y reintenta."""
    texto = "\n".join(f"{i}. {t}" for i, t in lote)
    for n in range(intentos):
        try:
            r = client.models.generate_content(
                model=MODELO_LLM,
                contents=PROMPT.format(titulares=texto),
                config={"response_mime_type": "application/json", "temperature": 0},
            )
            return json.loads(r.text)
        except errors.APIError as e:
            if e.code not in (429, 503):
                raise
            espera = 60 if e.code == 429 else 15 * (n + 1)
            print(f"  Error {e.code}, reintento en {espera} s...")
            time.sleep(espera)
    raise RuntimeError("Gemini sigue sin responder: espera unos minutos y vuelve a ejecutar la celda")

pares = list(enumerate(df["titular"]))
if "resultados" not in globals():            # si re-ejecutas la celda, no repite lo ya etiquetado
    resultados = []
hechos = {d["id"] for d in resultados}
for k in range(0, len(pares), 25):          # 25 titulares por llamada
    lote = [(i, t) for i, t in pares[k:k+25] if i not in hechos]
    if not lote:
        continue
    print(f"Lote {k // 25 + 1} de {-(-len(pares) // 25)}")
    nuevos = etiquetar_lote(lote)
    resultados += nuevos
    hechos |= {d["id"] for d in nuevos}
    time.sleep(5)                            # respeta el límite de la capa gratuita

res = pd.DataFrame(resultados).set_index("id")
etq = pd.DataFrame({
    "titular": df["titular"],
    "etiqueta_llm": res["etiqueta"].reindex(df.index),
    "confianza": res["confianza"].reindex(df.index),
    "modelo_llm": MODELO_LLM,
    "fecha": date.today().isoformat(),
})
etq.to_csv("datos/etiquetas_llm.csv", index=False, encoding="utf-8")
etq["etiqueta_llm"].value_counts(dropna=False)

Lote 8 de 16
Lote 9 de 16
Lote 10 de 16
Lote 11 de 16
Lote 12 de 16
Lote 13 de 16
Lote 14 de 16
Lote 15 de 16
Lote 16 de 16


etiqueta_llm
politica              106
deportes              100
ciencia_tecnologia     97
economia               97
Name: count, dtype: int64

### 3.1 Auditoría humana de 20 casos
Se eligen 20 titulares al azar (`random_state=42`). Primero se etiquetan a mano, sin ver la predicción del LLM, y luego se comparan.

In [8]:
pd.set_option("display.max_colwidth", None)        # muestra los titulares completos
audit = etq.sample(20, random_state=42)[["titular", "etiqueta_llm"]]
audit[["titular"]]                                  # oculta la etiqueta del LLM para no influir

,titular
209,Petroperú: proyecto de JP busca impedir que sus activos pasen al régimen de inversión privada
280,Kevin Quevedo pidió una nueva oportunidad en Alianza Lima: ¿qué debe hacer para volver a ser considerado por Pablo Guede?
33,"Cuando muere un elefante, su cadáver transforma la naturaleza de manera sorprendente, incluso después de varios años"
210,Una película biodegradable de seda de abeja podría transformar el tratamiento de las heridas crónicas
93,Detienen en Venezuela a cabecilla de la banda que mató a “China Polo”
84,Predictamen de la Comisión de Constitución busca incorporar 8 puntos nuevos de las facultades aprobadas por diputados
329,Cyber Wow 2026: Alu confirma descuentos de hasta 45% en cuatro líneas del 20 al 23 de abril
94,"Por primera vez, un planeta rocoso potencialmente habitable muestra signos de tener atmósfera"
266,"Una adolescente cavó un estanque en el patio de su casa para un proyecto escolar; años después, el lugar se convirtió en un ecosistema de vida silvestre"
126,Jurados Electorales Especiales iniciaron este lunes 12 de octubre el recuento de votos en todo el país


In [9]:
audit["etiqueta_humana"] = [
    "politica",   #  1. Petroperú: proyecto de JP busca impedir que sus activos pasen...
    "deportes",   #  2. Kevin Quevedo pidió una nueva oportunidad en Alianza Lima...
    "ciencia_tecnologia",   #  3. Cuando muere un elefante, su cadáver transforma la naturaleza...
    "ciencia_tecnologia",   #  4. Una película biodegradable de seda de abeja podría transformar...
    "politica",   #  5. Detienen en Venezuela a cabecilla de la banda que mató a “China Polo”
    "politica",   #  6. Predictamen de la Comisión de Constitución busca incorporar 8 puntos...
    "economia",   #  7. Cyber Wow 2026: Alu confirma descuentos de hasta 45%...
    "ciencia_tecnologia",   #  8. Por primera vez, un planeta rocoso potencialmente habitable...
    "ciencia_tecnologia",   #  9. Una adolescente cavó un estanque en el patio de su casa...
    "politica",   # 10. Jurados Electorales Especiales iniciaron el recuento de votos...
    "deportes",   # 11. PSG - Barcelona: fecha, hora y canal de TV...
    "deportes",   # 12. Jugó en el Barcelona, fue considerado el mejor portero de Alemania...
    "ciencia_tecnologia",   # 13. La Antártida se encuentra cubierta de hielo de hasta 4 km...
    "deportes",   # 14. Gino Vegas sobre su relación con Cenaida Uribe...
    "deportes",   # 15. Manchester City - PSG: fecha, hora y canal de TV...
    "politica",   # 16. Solicitan la destitución de Roberto Burneo...
    "economia",   # 17. Petroperú acelera crédito de US$475 millones...
    "economia",   # 18. ¿Cómo saber si tengo deudas por impuesto predial o arbitrios?...
    "politica",   # 19. Jorge Nieto a Keiko Fujimori: “En vez de amenazar, gobierne...”
    "ciencia_tecnologia",   # 20. Este imponente animal es el pez más grande del mundo...
]
validas = {"politica", "economia", "deportes", "ciencia_tecnologia"}
assert set(audit["etiqueta_humana"]) <= validas, "Hay una etiqueta vacía o mal escrita"

In [10]:
audit["coincide"] = audit["etiqueta_llm"] == audit["etiqueta_humana"]
acuerdo = int(audit["coincide"].sum())
print(f"Acuerdo LLM-Humano: {acuerdo}/20 ({acuerdo / 20:.0%})")
audit[~audit["coincide"]]          # los casos en que no coinciden

Acuerdo LLM-Humano: 20/20 (100%)


,titular,etiqueta_llm,etiqueta_humana,coincide


**Acuerdo LLM-Humano: 20/20 (100%).** En los 20 titulares auditados coincidí con la etiqueta de Gemini.

### 3.2 Etiquetas del LLM en cada split

In [11]:
df["etiqueta_llm"] = etq["etiqueta_llm"]
train, val, test = df.loc[train.index], df.loc[val.index], df.loc[test.index]

## 4. Fine-tuning de DistilBERT (paso 4 del enunciado)
`distilbert-base-multilingual-cased` entrenado con las etiquetas del LLM: 3 épocas, batch 16, evaluación por época y F1 macro para elegir el mejor modelo. Se guarda en `modelo/`.

In [12]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

MODEL = "distilbert-base-multilingual-cased"   # o "dccuchile/bert-base-spanish-wwm-cased"
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
tok = AutoTokenizer.from_pretrained(MODEL)

def a_dataset(d, col):
    ds = Dataset.from_dict({"text": d["titular"].tolist(),
                            "label": d[col].map(label2id).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

ds_train = a_dataset(train, "etiqueta_llm")   # supervisión débil
ds_val   = a_dataset(val, "etiqueta_llm")
ds_test  = a_dataset(test, "etiqueta")        # referencia: sección original

def metricas(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "f1_macro": f1_score(p.label_ids, pred, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="modelo",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    seed=42,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  processing_class=tok, compute_metrics=metricas)
trainer.train()
trainer.save_model("modelo")
tok.save_pretrained("modelo")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 18050.89it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/envs/tarea1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as t

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.676370,0.900000,0.900545
2,No log,0.417685,0.866667,0.866574
3,No log,0.309484,0.916667,0.919070


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.40it/s]
/opt/anaconda3/envs/tarea1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.60it/s]
/opt/anaconda3/envs/tarea1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.10it/s]


('modelo/tokenizer_config.json', 'modelo/tokenizer.json')